In [ ]:
# 1.ติดตั้ง library
!pip install -q transformers datasets accelerate pandas scikit-learn

In [1]:
# 2. โหลด dataset (ถ้าคุณรันส่วนสร้าง dataset ด้านบนแล้ว ข้ามได้)
# ==================================================
import pandas as pd
df = pd.read_csv("sample logs dataset.csv")
df.head()

,log,log_class,label
0,2025-12-20T10:09:04.341724Z 18 Query: sh...,Normal,0
1,2025-12-20T10:42:47.606339Z 21 Query: se...,Anormaly,1
2,2025-12-20T10:26:43.170481Z 64 Query: dr...,Anormaly,1
3,2025-12-20T10:27:12.777726Z 25 Query: us...,Normal,0
4,2025-12-20T10:27:01.995054Z 64 Query: ex...,Anormaly,1


In [2]:
# ==================================================
# 3. โหลด BERT และ tokenizer
# ==================================================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# ==================================================
# 4. Preprocess dataset
# ==================================================
# แยกเฉพาะส่วนหลัง timestamp และ thread ID (เช่น "Query: show databases")
def extract_command(log_line):
    parts = log_line.split()
    if len(parts) >= 4:
        return " ".join(parts[2:])  # ข้าม timestamp กับ ID
    return log_line

df["text"] = df["log"].apply(extract_command)

# แบ่ง train/val
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label"]])

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [ ]:
!pip install --upgrade transformers datasets accelerate

In [8]:
# ==================================================
# 5. ตั้งค่าการ train
# ==================================================

training_args = TrainingArguments(
    output_dir="Fine Tuned Model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,

    save_strategy="epoch",
    load_best_model_at_end=False,   # ❌ ปิด

    logging_dir="./logs",
    report_to="none",
)


In [10]:
# 5.1. แก้ data
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")


In [11]:
# ==================================================
# 6. Train!
# ==================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,   # ⭐ สำคัญมาก
)


print("🚀 เริ่ม fine-tuning...")
trainer.train()


🚀 เริ่ม fine-tuning...


Step,Training Loss


TrainOutput(global_step=300, training_loss=0.08488931655883789, metrics={'train_runtime': 62.5506, 'train_samples_per_second': 76.738, 'train_steps_per_second': 4.796, 'total_flos': 56725099091520.0, 'train_loss': 0.08488931655883789, 'epoch': 3.0})

In [12]:
# ==================================================
# 7. บันทึกโมเดล
# ==================================================
trainer.save_model("./mysql-anomaly-bert-final")
tokenizer.save_pretrained("./mysql-anomaly-bert-final")

print("✅ บันทึกโมเดลที่ fine-tune แล้วที่: ./mysql-anomaly-bert-final")

✅ บันทึกโมเดลที่ fine-tune แล้วที่: ./mysql-anomaly-bert-final


In [13]:
# บีบอัดโฟลเดอร์โมเดล
!zip -r mysql-anomaly-bert-final.zip mysql-anomaly-bert-final

# ดาวน์โหลด (คลิกที่ลิงก์ที่แสดง)
from google.colab import files
files.download("mysql-anomaly-bert-final.zip")

  adding: mysql-anomaly-bert-final/ (stored 0%)
  adding: mysql-anomaly-bert-final/special_tokens_map.json (deflated 42%)
  adding: mysql-anomaly-bert-final/tokenizer_config.json (deflated 75%)
  adding: mysql-anomaly-bert-final/training_args.bin (deflated 53%)
  adding: mysql-anomaly-bert-final/config.json (deflated 49%)
  adding: mysql-anomaly-bert-final/vocab.txt (deflated 53%)
  adding: mysql-anomaly-bert-final/model.safetensors (deflated 7%)
  adding: mysql-anomaly-bert-final/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Inference ทดสอบ Model หลังจาก finetune

In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# โหลดโมเดลท้องถิ่น
model_path = "/content/mysql-anomaly-bert-final"  # แก้ path ให้ตรง
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)


In [15]:
# ทำนาย log ใหม่
def predict_anomaly(log_line):
    # ดึง command (เหมือนตอน train)
    parts = log_line.split()
    text = " ".join(parts[2:]) if len(parts) >= 4 else log_line

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=-1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]
    return "anomaly" if pred == 1 else "normal", prob

In [25]:
# ตัวอย่าง (row 13 แปลกๆ)
log = "2025-12-20T10:39:41.931343Z       17 Query: select load_file('/etc/passwd')"
result, confidence = predict_anomaly(log)
print(f"Prediction: {result}, Confidence: {confidence}")

Prediction: anomaly, Confidence: [0.00011150370846735314, 0.99988853931427]


In [27]:
log = "2025-12-20T10:15:54.750047Z       30 Query: delete from products"
result, confidence = predict_anomaly(log)
print(f"Prediction: {result}, Confidence: {confidence}")

Prediction: anomaly, Confidence: [0.00013319375284481794, 0.9998668432235718]


In [26]:
log = "2025-12-20T10:09:04.341724Z       18 Query: show databases"
result, confidence = predict_anomaly(log)
print(f"Prediction: {result}, Confidence: {confidence}")

Prediction: normal, Confidence: [0.9998780488967896, 0.0001219139012391679]


In [28]:
log = "2025-12-20T10:09:04.341724Z       18 Query: show databases"
result, confidence = predict_anomaly(log)
print(result)

normal
